# LinkedIn Card — Prithvi Burn Scar

Genera la figura compuesta (Top-4 patches + Training curve + Métricas) 
estilo portfolio para LinkedIn.  
**Runtime**: Colab CPU o GPU — aprox. 5-10 min en CPU, 2 min en T4.

In [ ]:
# ── Install dependencies (restarts the kernel automatically if needed) ──
import subprocess, sys, os

try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    print(f'Installing ({e})...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==2.0.2', '--force-reinstall'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
# ── Imports + Drive + Device ───────────────────────────────────────────────────
import warnings
import subprocess
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
from terratorch.registry import BACKBONE_REGISTRY

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Paths + copy patches a /content/ ─────────────────────────────────────────
BASE    = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
CKPT    = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
RESULTS = BASE / 'results'

LOCAL = Path('/content/patches')
LOCAL.mkdir(exist_ok=True)
if not (LOCAL / 'images').exists():
    print('Copying patches to /content/ ...')
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/images'),     str(LOCAL)], check=True)
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks_dnbr'), str(LOCAL)], check=True)
    subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks'),      str(LOCAL)], check=True)
    print('Done.')

IMG_DIR   = LOCAL / 'images'
DNBR_DIR  = LOCAL / 'masks_dnbr'
FIRMS_DIR = LOCAL / 'masks'

img_paths   = sorted(IMG_DIR.glob('*.npy'))
dnbr_paths  = sorted(DNBR_DIR.glob('*.npy'))
firms_paths = sorted(FIRMS_DIR.glob('*.npy'))

assert CKPT.exists(), f'Checkpoint no encontrado: {CKPT}'
print(f'Patches: {len(img_paths)}  |  Checkpoint: OK ({CKPT.stat().st_size/1e6:.0f} MB)')

In [ ]:
# ── Model architecture (identical to notebook 07) ─────────────────────────
class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side
    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]
        B, L, D = x.shape
        return x.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        feats = self.backbone(x.unsqueeze(2))
        return self.decoder(self.neck(feats))


print('Loading Prithvi backbone...')
backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
for p in backbone.parameters():
    p.requires_grad = False

model = PrithviSegModel(backbone).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()
print('Model loaded.')

In [ ]:
# ── Scan patches → inference → rank by IoU ────────────────────────
BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2   # crop offset: 16
THRESHOLD = 0.5

# Visual quality filters:
#   - dnbr_rate entre 0.15 y 0.78  → mezcla real de quemado/no quemado
#   - píxeles verde (banda 1) en zona no quemada > umbral → vegetación visible
DNR_LO, DNR_HI = 0.15, 0.78
MIN_GREEN = 500
MIN_UNBURNED_PX = 300

ranked = []
for ip, fp, dp in tqdm(zip(img_paths, firms_paths, dnbr_paths), total=len(img_paths)):
    dnbr_raw  = np.load(dp, mmap_mode='r')
    dnbr_rate = float(dnbr_raw.mean())
    if not (DNR_LO < dnbr_rate < DNR_HI):
        continue

    raw = np.load(ip, mmap_mode='r').astype(np.float32)    # (11, 256, 256)
    unburned_px = raw[1][dnbr_raw == 0]
    if len(unburned_px) < MIN_UNBURNED_PX or float(unburned_px.mean()) < MIN_GREEN:
        continue

    # Preprocesar para Prithvi
    img_norm = raw[BAND_IDX] / 10000.0
    img_norm = (img_norm - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
    img_crop  = img_norm[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
    dnbr_crop = dnbr_raw[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()

    tensor = torch.from_numpy(img_crop).unsqueeze(0).to(DEVICE)  # (1,6,224,224)
    with torch.no_grad():
        prob = model(tensor).sigmoid().squeeze().cpu().numpy()    # (224,224)
    pred = (prob > THRESHOLD).astype(np.float32)

    tp  = (pred * dnbr_crop).sum()
    fp2 = (pred * (1 - dnbr_crop)).sum()
    fn  = ((1 - pred) * dnbr_crop).sum()
    patch_iou = float(tp / (tp + fp2 + fn + 1e-6))

    ranked.append({
        'iou':  patch_iou,
        'img':  raw.copy(),       # (11,256,256) completo para RGB
        'dnbr': dnbr_crop,        # (224,224)
        'pred': pred,             # (224,224)
    })

ranked.sort(key=lambda x: x['iou'], reverse=True)
top4 = ranked[:4]
print(f'Candidatos con fuego: {len(ranked)}')
print(f'Top-4 IoUs: {[f"{r[\"iou\"]:.3f}" for r in top4]}')

In [ ]:
# ── LinkedIn card — layout idéntico al proyecto Forest ────────────────────────

# Métricas globales del evaluation (notebook 07 + 06_evaluate)
iou_global  = 0.4198
f1_global   = 0.5913
prec_global = 0.4968
rec_global  = 0.7303
N_VAL       = 1137   # 20% de 5,687 patches (VAL_FRAC=0.2, SEED=42)

def make_rgb(raw_arr):
    """raw_arr: (11,256,256) uint16-range float32 → RGB crop (224,224,3) [0,1]"""
    crop = raw_arr[:, T:T+IMG_SIZE, T:T+IMG_SIZE] / 10000.0
    def nb(b, p=1):
        v = b[b > 0]
        lo, hi = np.percentile(v, [p, 100-p]) if len(v) > 0 else (0, 1)
        return np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)
    return np.dstack([nb(crop[2]), nb(crop[1]), nb(crop[0])])


fig = plt.figure(figsize=(12, 6.28), facecolor='white')

fig.text(0.5, 0.97,
    'Wildfire Burn Scar Detection from Sentinel-2 Satellite Imagery',
    ha='center', va='top', fontsize=13, fontweight='bold', color='#111111')
fig.text(0.5, 0.915,
    'PyTorch  |  Prithvi-100M (IBM/NASA)  |  6-channel Sentinel-2 L2A  |  dNBR labels  |  Corrientes, Argentina 2022',
    ha='center', va='top', fontsize=9, color='#555555')

gs_outer = gridspec.GridSpec(2, 1, figure=fig,
                             height_ratios=[0.88, 0.12],
                             hspace=0.05, left=0.02, right=0.98,
                             top=0.83, bottom=0.02)

gs_main = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=gs_outer[0],
                                           width_ratios=[1.5, 1.0], wspace=0.08)

# ── Left column: header + 2x2 patch grid ────────────────────────
gs_left = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[0],
                                           height_ratios=[0.08, 0.92], hspace=0.03)
ax_left_hdr = fig.add_subplot(gs_left[0])
ax_left_hdr.axis('off')
ax_left_hdr.text(0.5, 0.2,
    'Top-4 detected patches from validation set  (ranked by per-patch IoU)',
    ha='center', va='bottom', fontsize=8.5, color='#444444', style='italic',
    transform=ax_left_hdr.transAxes)

gs_grid = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs_left[1],
                                           hspace=0.04, wspace=0.04)
rank_labels = ['BEST', '#2', '#3', '#4']

for idx, r in enumerate(top4):
    row, col = divmod(idx, 2)
    ax = fig.add_subplot(gs_grid[row, col])

    rgb2 = make_rgb(r['img'])
    pred_b = r['pred']
    gt     = r['dnbr']

    err = np.zeros((*pred_b.shape, 3))
    err[(pred_b==1) & (gt==1)] = [0.0,  0.82, 0.0 ]
    err[(pred_b==1) & (gt==0)] = [1.0,  0.55, 0.0 ]
    err[(pred_b==0) & (gt==1)] = [0.85, 0.10, 0.10]
    err_a = np.where(err.sum(axis=2) > 0, 0.85, 0.0)

    ax.imshow(rgb2)
    ax.imshow(np.dstack([err, err_a]))
    ax.set_xticks([]); ax.set_yticks([])

    if idx == 0:
        for s in ['top', 'bottom', 'left', 'right']:
            ax.spines[s].set_visible(True)
            ax.spines[s].set_color('#2ea44f')
            ax.spines[s].set_linewidth(3)
        ax.text(4, 14, f'BEST  |  Patch IoU = {r["iou"]:.3f}', color='white', fontsize=8.5,
                fontweight='bold', bbox={'fc': '#2ea44f', 'alpha': 0.95, 'pad': 3})
    else:
        for s in ax.spines.values():
            s.set_visible(False)
        ax.text(4, 14, f'{rank_labels[idx]}  |  Patch IoU = {r["iou"]:.3f}', color='white',
                fontsize=8, bbox={'fc': '#1e3a5f', 'alpha': 0.85, 'pad': 3})

    if idx == 3:
        ax.legend(handles=[
            mpatches.Patch(color=[0.0, 0.82, 0.0],  label='True Positive'),
            mpatches.Patch(color=[1.0, 0.55, 0.0],  label='False Positive'),
            mpatches.Patch(color=[0.85, 0.10, 0.10], label='False Negative'),
        ], loc='lower right', fontsize=7, framealpha=0.85, handlelength=1)

# ── Columna derecha: spacer + curva training + métricas ──────────────────────
gs_right_wrapper = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[1],
                                                    height_ratios=[0.08, 0.92], hspace=0.0)
fig.add_subplot(gs_right_wrapper[0]).axis('off')   # spacer para alinear con header izquierdo
gs_right = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_right_wrapper[1], hspace=0.55)

# Training curve — embebemos el PNG guardado (épocas 21-40)
ax_curve = fig.add_subplot(gs_right[0])
curve_img = mpimg.imread(str(RESULTS / 'training_curves_prithvi_burn_scar.png'))
# El PNG tiene dos subplots (loss + IoU). Nos quedamos con la mitad derecha (IoU).
h, w = curve_img.shape[:2]
ax_curve.imshow(curve_img[:, w//2:, :] if curve_img.ndim == 3 else curve_img[:, w//2:])
ax_curve.axis('off')
ax_curve.set_title('Training convergence  (epochs 21–40, Colab A100)', fontsize=8.5, fontweight='bold', pad=4)

# Métricas globales — bar chart horizontal
ax_bars = fig.add_subplot(gs_right[1])
metric_names  = ['Recall', 'Precision', 'F1', 'IoU']
metric_values = [rec_global, prec_global, f1_global, iou_global]
bars = ax_bars.barh(metric_names, metric_values, color='#4a5568', height=0.5)
for bar, val in zip(bars, metric_values):
    ax_bars.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=8, fontweight='bold', color='#1e3a5f')
ax_bars.set_xlim(0, 0.90)
ax_bars.set_title(f'Global validation metrics  (n={N_VAL:,} patches)', fontsize=8.5, fontweight='bold')
ax_bars.tick_params(labelsize=8)
ax_bars.grid(axis='x', alpha=0.3)
ax_bars.spines['top'].set_visible(False)
ax_bars.spines['right'].set_visible(False)

# ── Footer ───────────────────────────────────────────────────────────────────
ax_foot = fig.add_subplot(gs_outer[1])
ax_foot.axis('off')
ax_foot.text(0.5, 0.5,
    'Prithvi-EO-1.0-100M (IBM/NASA)  |  FPN decoder  |  '
    'Labels: dNBR > 0.10  |  Corrientes, Argentina 2022  |  ~900,000 ha burned',
    ha='center', va='center', fontsize=9, color='#666666',
    transform=ax_foot.transAxes)

out_path = RESULTS / 'linkedin_card_prithvi.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Guardado: {out_path}')